# Data Quality
This notebook contains the code for data quality checks, including missing values, duplicates, types, inconsistent labels and EDA visualizations.

### Imports

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.data import load_raw, quality_report, label_issues, rows_to_drop, multiclass_label, preprocess
from src.splits import save_splits, load_split

## General quality check

In [ ]:
df = load_raw()
df.info()
quality_report(df).to_frame("count")

**Findings**
- 10000 rows, 14 columns, correct types.
- No missing values, as expected, no duplicate rows, no duplicate UDIs.
- `Type` always matches the first letter of `Product ID`. There are no negative values, and process temperature is never below air temperature.
> No imputation or cleaning of the features is needed.

## Label consistency

In [ ]:
issues = label_issues(df)
display(issues["reason"].value_counts())
issues

**Findings**

`Machine failure` should be 1 if and only if at least one mode is active. That rule is broken in two ways:
- **9 rows `failure_no_mode`:** a failure with no mode. The failure label is valid, but there is no mode to assign.
- **18 rows `mode_no_failure`:** RNF active but `Machine failure = 0`. These are the only true inconsistencies.

Separately, **24 rows `multiple_modes`** have more than one mode at once. They are not errors, because the modes are independent and can happen together.

## Multiclass rows and final classes

In [ ]:
multi_kept = df.drop(rows_to_drop(df, "multi"))
print(f"Multiclass: dropped {len(df) - len(multi_kept)} rows, kept {len(multi_kept)}")
multiclass_label(multi_kept).value_counts()

**Decisions**
- **Binary:** keep all 10000 rows and use `Machine failure` as given. The 18 `mode_no_failure` rows stay as no failure.
- **Multiclass (No failure, TWF, HDF, PWF, OSF):** drop 27 rows with no learnable label, the 9 `failure_no_mode` and the 18 RNF only. RNF is excluded because it is random.
- **Multi-mode rows:** instead of having both modes, the rarest one is the one given (TWF < OSF < PWF < HDF). This keeps 24 real failures and keeps these hard cases in the test set. The cost is that predicting the other true mode counts as an error, we will check this in the error analysis.

**Final classes**: 9643 No failure, 106 HDF, 95 OSF, 83 PWF, 46 TWF.

## Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd 
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
class_order = ["No failure", "TWF", "HDF", "PWF", "OSF"]
class_labels = multiclass_label(multi_kept)
eda_df = multi_kept.assign(Class=class_labels)
class_counts = class_labels.value_counts().reindex(class_order, fill_value=0)
project_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
figures_dir = project_root / "results" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
multiclass_feature_summary = pd.DataFrame()

for class_name in class_order:
    class_data = eda_df.loc[eda_df["Class"] == class_name, feature_cols]

    multiclass_feature_summary[f"{class_name}_mean"] = class_data.mean()
    multiclass_feature_summary[f"{class_name}_median"] = class_data.median()
    multiclass_feature_summary[f"{class_name}_std"] = class_data.std()

display(multiclass_feature_summary.round(2))

## Feature Distribuitions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.flat, feature_cols):
    sns.histplot(data=df, x=feature, bins=30, ax=ax, color="steelblue")
    ax.set_title(feature)
axes.flat[-1].set_visible(False)
fig.suptitle("Process feature distributions", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "eda_feature_histograms.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Multiclass Target Compoition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

wedges, texts = ax.pie(
    class_counts,
    startangle=90,
    counterclock=False,
)
labels = [
    f"{label}: {count / class_counts.sum() * 100:.1f}%"
    for label, count in class_counts.items()
]

ax.legend(
    wedges,
    labels,
    title="Classes",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

ax.set_title("Five-class target composition")
fig.savefig(figures_dir / "eda_class_pie.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


## Feature Distributions Across Failure Classes

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flat, feature_cols):
    sns.boxplot(
        data=eda_df,
        x="Class",
        y=feature,
        order=class_order,
        showfliers=False,
        ax=ax,
    )
    ax.set_title(feature)
    ax.tick_params(axis="x", labelrotation=25)
axes.flat[-1].set_visible(False)
fig.suptitle("Process measurements by cleaned class (fliers hidden)", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "eda_boxplots_by_class.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


##  Feature Correlation

In [ ]:
correlations = df[feature_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    correlations,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Pearson correlation between process features")
fig.tight_layout()
fig.savefig(figures_dir / "eda_feature_correlations.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Class Imbalance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=class_counts.values, y=class_counts.index, color="steelblue", ax=ax)
ax.set_xscale("log")
ax.set_xlabel("Number of rows (log scale)")
ax.set_ylabel("Class")
ax.set_title("Class imbalance in the cleaned multiclass target")
for y, count in enumerate(class_counts.values):
    ax.text(count * 1.08, y, f"{count:,}", va="center")
ax.set_xlim(right=class_counts.max() * 1.5)
fig.tight_layout()
fig.savefig(figures_dir / "eda_class_imbalance.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
imbalance_table = pd.DataFrame(
    {
        "count": class_counts,
        "share_percent": class_counts / class_counts.sum() * 100,
    }
)
display(imbalance_table)
print(f"Majority-to-minority ratio: {class_counts.max() / class_counts.min():.1f}:1")

**Findings**

- TWF has the highest median tool wear (215 min), clearly separating it from the other classes.
- HDF has the highest median air temperature (302.4 K) and process temperature (310.7 K).
- PWF has the greatest variability in rotational speed and torque with big distribution.
- OSF has relatively high median torque (57 Nm) and tool wear (206 min).
- Feature overlap: TWF shows similar torque and rotational speed distributions to normal operation, making individual feature separation difficult.
- Strong correlations: Air and process temperature are positively correlated (0.88), while rotational speed and torque are negatively correlated (-0.88).
- Different distributions: Rotational speed is right-skewed, torque is approximately bell-shaped, and tool wear is relatively evenly distributed.

## Binary Target Balance

In [ ]:
binary_counts = df["Machine failure"].value_counts().reindex([0, 1], fill_value=0)
binary_summary = pd.DataFrame(
    {
        "class": ["No failure", "Failure"],
        "count": binary_counts.to_numpy(),
        "share_percent": binary_counts.to_numpy() / len(df) * 100,
    }
)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(binary_summary["class"], binary_summary["count"], color=["steelblue", "indianred"])
ax.set_ylabel("Number of rows")
ax.set_title("Binary target balance")
ax.set_ylim(0, binary_summary["count"].max() * 1.15)
for bar, count, share in zip(bars, binary_summary["count"], binary_summary["share_percent"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{count:,} ({share:.2f}%)", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(figures_dir / "eda_binary_class_balance.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Failure by Product

In [ ]:
type_failure = df.groupby("Type", observed=True)["Machine failure"].agg(
    total_rows="size",
    failures="sum",
)
type_failure["failure_rate_percent"] = type_failure["failures"] / type_failure["total_rows"] * 100

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(type_failure.index, type_failure["failure_rate_percent"], color="darkorange")
ax.set_xlabel("Product type")
ax.set_ylabel("Failure rate (%)")
ax.set_title("Observed failure rate by product type")
for bar, failures, total in zip(bars, type_failure["failures"], type_failure["total_rows"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{failures} / {total}", ha="center", va="bottom")
ax.set_ylim(0, type_failure["failure_rate_percent"].max() * 1.2)
fig.tight_layout()
fig.savefig(figures_dir / "eda_binary_failure_rate_by_type.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Process Meassurements 

In [ ]:
no_failure = df.loc[df["Machine failure"] == 0, feature_cols]
failure = df.loc[df["Machine failure"] == 1, feature_cols]
binary_feature_summary = pd.DataFrame(
    {
        "no_failure_mean": no_failure.mean(),
        "failure_mean": failure.mean(),
        "mean_difference": failure.mean() - no_failure.mean(),
        "no_failure_median": no_failure.median(),
        "failure_median": failure.median(),
    }
)
display(binary_feature_summary.round(2))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flat, feature_cols):
    sns.boxplot(data=df, x="Machine failure", y=feature, order=[0, 1], showfliers=False, ax=ax)
    ax.set_title(feature)
    ax.set_xlabel("Target")
    ax.set_xticks([0, 1], ["No failure", "Failure"])
axes.flat[-1].set_visible(False)
fig.suptitle("Process measurements by binary target (fliers hidden)", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "eda_binary_features_by_target.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

**Findings**

- There are 9,661 no-failure rows (96.61%) and 339 failure rows (3.39%). A classifier that always predicts no failure would therefore reach 96.61% accuracy while detecting none of the failures, so accuracy alone is not informative here; failure-class recall, precision, and F1 should also be considered.
- Failure rates differ across product types in this dataset (L: 3.92%, M: 2.77%, H: 2.09%). These are descriptive differences, not evidence that product type causes failure.
- The failure rows have higher mean torque (50.17 vs. 39.63 Nm) and tool wear (143.78 vs. 106.69 min), and lower mean rotational speed (1,496.49 vs. 1,540.26 rpm). The temperature means differ less. The box plots show the distributions and overlap; outliers are hidden for readability, not removed from the analysis.
- These comparisons are exploratory and do not establish causal relationships. The plots and summary statistics use the full dataset to describe the target; model evaluation must continue to use the project's separate stratified train/test split.

In [ ]:
print(f"Figures saved to: {figures_dir.resolve()}")